# Malaika — Fine-Tune Gemma 4 **E2B** with Unsloth (Edge/Phone)

**This notebook fine-tunes the smaller E2B model for phone deployment.**

- **E2B** = 2.3B active params, 2.58GB disk, runs on phone via LiteRT-LM
- Same pipeline as E4B: ICBHI Audio → mel-spectrogram → Gemma 4 vision QLoRA
- Binary classification (normal vs abnormal) — aligns with IMCI protocol

**Run on Kaggle with T4 GPU.** Add `vbookshelf/respiratory-sound-database` as dataset input.

In [ ]:
%%capture
# Cell 1: Install Unsloth (restart runtime after this cell)
!pip install unsloth
!pip install -q --no-deps trl datasets librosa soundfile Pillow

In [ ]:
# Cell 2: Authenticate
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))
print("Authenticated")

In [ ]:
# Cell 3: Imports + GPU check
import json, os, random, re, time
from collections import Counter
from pathlib import Path

os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

import numpy as np
import torch
import librosa
from PIL import Image

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Load ICBHI dataset (auto-detect Kaggle path)
import glob as _glob
_candidates = [
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database",
    "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database",
    "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database",
]
ICBHI_PATH = None
for _c in _candidates:
    if Path(_c).exists():
        ICBHI_PATH = Path(_c)
        break

if ICBHI_PATH is None:
    _found = _glob.glob("/kaggle/input/**/audio_and_txt_files", recursive=True)
    if _found:
        ICBHI_PATH = Path(_found[0]).parent

if ICBHI_PATH is not None:
    audio_dir = ICBHI_PATH / "audio_and_txt_files"
    if audio_dir.exists():
        audio_files = list(audio_dir.glob("*.wav"))
        print(f"ICBHI dataset found at: {ICBHI_PATH}")
        print(f"Audio files: {len(audio_files)}")
    else:
        print(f"Found {ICBHI_PATH} but no audio_and_txt_files/ inside")
        audio_files = []
else:
    print("ERROR: Dataset not found. Add 'respiratory-sound-database' by vbookshelf via Add Data")
    print("Debug — listing /kaggle/input/:")
    !find /kaggle/input/ -maxdepth 4 -type d
    audio_files = []

In [ ]:
# Cell 5: Parse annotations
def parse_icbhi_annotations(audio_dir):
    records = []
    for txt_file in sorted(audio_dir.glob("*.txt")):
        wav_file = txt_file.with_suffix(".wav")
        if not wav_file.exists(): continue
        parts = txt_file.stem.split("_")
        patient_id = parts[0] if parts else "unknown"
        has_crackle, has_wheeze = False, False
        with open(txt_file) as f:
            for line in f:
                fields = line.strip().split()
                if len(fields) >= 4:
                    if int(fields[2]) == 1: has_crackle = True
                    if int(fields[3]) == 1: has_wheeze = True
        if has_crackle and has_wheeze: label = "both"
        elif has_crackle: label = "crackle"
        elif has_wheeze: label = "wheeze"
        else: label = "normal"
        records.append({"audio_path": str(wav_file), "patient_id": patient_id,
            "label": label, "has_crackle": has_crackle, "has_wheeze": has_wheeze})
    return records

if audio_files:
    records = parse_icbhi_annotations(audio_dir)
    print(f"Parsed {len(records)} recordings (original 4-class):")
    for label, count in sorted(Counter(r["label"] for r in records).items()):
        print(f"  {label}: {count}")

In [ ]:
# Cell 6: Generate spectrograms
SPEC_DIR = Path("/kaggle/working/spectrograms")
SPEC_DIR.mkdir(exist_ok=True)

SPEC_W, SPEC_H = 512, 256

def audio_to_spec(audio_path, output_path):
    try:
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
        if len(y) == 0: return False
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=50, fmax=4000)
        db = librosa.power_to_db(mel, ref=np.max)
        mn, mx = db.min(), db.max()
        norm = ((db - mn) / (mx - mn) * 255).astype(np.uint8) if mx > mn else np.zeros_like(db, dtype=np.uint8)
        norm = np.flip(norm, axis=0)
        Image.fromarray(norm, mode="L").resize((SPEC_W, SPEC_H), Image.Resampling.LANCZOS).convert("RGB").save(str(output_path))
        return True
    except Exception as e:
        print(f"  Skip {Path(audio_path).name}: {e}")
        return False

if audio_files:
    print("Generating spectrograms...")
    spec_records = []
    for i, r in enumerate(records):
        sp = SPEC_DIR / f"{Path(r['audio_path']).stem}_spec.png"
        if audio_to_spec(r['audio_path'], sp):
            r["spectrogram_path"] = str(sp)
            spec_records.append(r)
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(records)}")
    print(f"Generated {len(spec_records)}/{len(records)} spectrograms")
    records = spec_records

In [ ]:
# Cell 7: BINARY classification — normal vs abnormal + class balancing
# IMCI protocol only needs: "are there abnormal breath sounds?" (not wheeze vs crackle)

INSTRUCTION = ("This is a mel-spectrogram of a child's breathing audio.\n"
    "Vertical: frequency (50-4000 Hz). Horizontal: time. Brightness: intensity.\n"
    "Are the breath sounds normal or abnormal?\n"
    'Respond with JSON: {"abnormal": true/false, "confidence": 0.0-1.0, "description": "brief reason"}')

SYSTEM_MSG = ("You are Malaika, a medical spectrogram analysis assistant. "
    "Detect abnormal breath sounds (wheeze, crackles, stridor) from spectrograms. "
    "Respond ONLY with valid JSON.")

def make_binary_response(r):
    is_abnormal = r["label"] != "normal"
    if not is_abnormal:
        desc = "Normal breath sounds. Clear airways, no adventitious sounds."
    else:
        parts = []
        if r["has_wheeze"]: parts.append("wheezing (continuous high-pitched sounds)")
        if r["has_crackle"]: parts.append("crackles (discontinuous popping sounds)")
        desc = "Abnormal: " + " and ".join(parts) + " detected."
    return json.dumps({"abnormal": is_abnormal, "confidence": 0.9, "description": desc})

if audio_files:
    # Create binary pairs
    pairs = [{"spectrogram_path": r["spectrogram_path"],
              "label": "abnormal" if r["label"] != "normal" else "normal",
              "original_label": r["label"],
              "instruction": INSTRUCTION,
              "response": make_binary_response(r)} for r in records]

    # Patient-level split
    pids = list(set(r["patient_id"] for r in records))
    random.seed(42); random.shuffle(pids)
    train_pats = set(pids[:int(len(pids)*0.8)])
    train_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] in train_pats]
    test_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] not in train_pats]

    # Class balancing
    label_counts = Counter(p["label"] for p in train_pairs)
    max_count = max(label_counts.values())
    print(f"Binary distribution before balancing: {dict(label_counts)}")

    balanced_pairs = []
    for label in label_counts:
        class_pairs = [p for p in train_pairs if p["label"] == label]
        oversampled = (class_pairs * (max_count // len(class_pairs) + 1))[:max_count]
        balanced_pairs.extend(oversampled)
    random.seed(42); random.shuffle(balanced_pairs)
    train_pairs = balanced_pairs

    label_counts_after = Counter(p["label"] for p in train_pairs)
    print(f"After balancing: {dict(label_counts_after)}")
    print(f"Train: {len(train_pairs)} | Test: {len(test_pairs)}")
    print(f"Test distribution: {dict(Counter(p['label'] for p in test_pairs))}")

In [ ]:
# Cell 8: Load E2B model with Unsloth (smaller — for phone/edge)
from unsloth import FastModel

# Skip Unsloth's telemetry — crashes on Kaggle due to HF 500/503/504 errors
import unsloth.models._utils as _u
import unsloth.models.vision as _v
_noop = lambda *a, **kw: None
_u.get_statistics = _noop
_v.get_statistics = _noop

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)

print(f"Model loaded (E2B) | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 9: Add LoRA adapter — SMALLER rank to prevent overfitting
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    random_state=3407,
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 10: Format training data
from datasets import Dataset

def format_for_training(pairs):
    formatted = []
    for p in pairs:
        formatted.append({"messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_MSG}]},
            {"role": "user", "content": [
                {"type": "image", "image": p["spectrogram_path"]},
                {"type": "text", "text": p["instruction"]},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": p["response"]}]},
        ]})
    return formatted

if audio_files:
    train_formatted = format_for_training(train_pairs)
    train_dataset = Dataset.from_list(train_formatted)
    print(f"Training dataset: {len(train_dataset)} examples")
else:
    dummy_path = SPEC_DIR / "dummy.png"
    if not dummy_path.exists():
        Image.fromarray(np.random.randint(0, 255, (SPEC_H, SPEC_W, 3), dtype=np.uint8)).save(str(dummy_path))
    train_dataset = Dataset.from_list([{"messages": [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_MSG}]},
        {"role": "user", "content": [
            {"type": "image", "image": str(dummy_path)},
            {"type": "text", "text": "Classify breath sounds."}]},
        {"role": "assistant", "content": [{"type": "text", "text":
            '{"abnormal": false, "confidence": 0.9, "description": "Normal"}'}]},
    ]}] * 10)
    print("Using dummy data")

In [ ]:
# Cell 11: Setup vision data collator + VERIFY images are loaded
from unsloth.trainer import UnslothVisionDataCollator
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("google/gemma-4-E2B-it")
data_collator = UnslothVisionDataCollator(model, processor)

# CRITICAL: verify pixel_values are present
sample = train_dataset[0]
batch = data_collator([sample])
has_pixel = "pixel_values" in batch
pixel_shape = tuple(batch["pixel_values"].shape) if has_pixel else "MISSING"
input_shape = tuple(batch["input_ids"].shape) if "input_ids" in batch else "MISSING"

print(f"pixel_values present: {has_pixel}")
print(f"pixel_values shape:   {pixel_shape}")
print(f"input_ids shape:      {input_shape}")
if has_pixel:
    print("CONFIRMED: Images ARE being processed during training")
else:
    print("STOP: Images NOT being processed. Training will be text-only and useless.")

In [ ]:
# Cell 12: Train — E2B needs MORE steps than E4B (slower learner)
# E4B converged at step 40. E2B still at loss=0.26 at step 100.
# Give it 300 steps with higher LR to fully converge.
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=processor.tokenizer,
    train_dataset=train_dataset,
    data_collator=data_collator,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=300,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.05,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        output_dir="/kaggle/working/breath_sound_lora",
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print("Starting E2B training (300 steps, lr=2e-4)...")
t0 = time.monotonic()
result = trainer.train()
train_time = time.monotonic() - t0
print(f"\nTraining complete in {train_time/60:.1f} min")
print(f"  Loss: {result.training_loss:.4f} | Steps: {result.global_step}")

In [ ]:
# Cell 13: Save adapter
ADAPTER_NAME = "/kaggle/working/malaika-breath-sounds-E2B-lora"

model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Adapter saved to {ADAPTER_NAME}/")

for f in sorted(Path(ADAPTER_NAME).glob("*")):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f} MB")

In [ ]:
# Cell 14: Evaluate — BINARY (normal vs abnormal)
def extract_json(text):
    cleaned = re.sub(r'```json\s*', '', text)
    cleaned = re.sub(r'```\s*$', '', cleaned)
    m = re.search(r'\{[^{}]*\}', cleaned)
    if m:
        return json.loads(m.group(0))
    return None

print("=" * 60)
print("EVALUATION — BINARY (normal vs abnormal)")
print("=" * 60)

if audio_files and test_pairs:
    correct, total_test = 0, 0
    tp, fp, tn, fn = 0, 0, 0, 0
    by_label = {}
    for pair in test_pairs[:50]:
        spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": pair["instruction"]}]}]
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=input_text, images=[spec_img], return_tensors="pt").to(model.device)
        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        label = pair["label"]
        expected_abnormal = (label == "abnormal")
        try:
            pred = extract_json(pred_text)
            exp = json.loads(pair["response"])
            if pred and pred.get("abnormal") == exp.get("abnormal"):
                correct += 1; status = "PASS"
            else:
                status = "FAIL"
            # Confusion matrix
            if pred:
                pred_abn = pred.get("abnormal", False)
                if pred_abn and expected_abnormal: tp += 1
                elif pred_abn and not expected_abnormal: fp += 1
                elif not pred_abn and not expected_abnormal: tn += 1
                elif not pred_abn and expected_abnormal: fn += 1
        except: status = "FAIL (parse)"
        total_test += 1
        by_label.setdefault(label, [0, 0])
        by_label[label][1] += 1
        if status == "PASS": by_label[label][0] += 1
        print(f"  {status} [{label:8s}] [{pair.get('original_label','?'):7s}] -- {pred_text[:90]}")

    acc = 100*correct/total_test if total_test else 0
    print(f"\nOverall accuracy: {correct}/{total_test} ({acc:.0f}%)")
    print("Per-class:")
    for label, (c, t) in sorted(by_label.items()):
        print(f"  {label}: {c}/{t} ({100*c/t:.0f}%)")
    print(f"\nConfusion matrix:")
    print(f"  TP={tp} FP={fp} TN={tn} FN={fn}")
    precision = tp/(tp+fp) if (tp+fp) > 0 else 0
    recall = tp/(tp+fn) if (tp+fn) > 0 else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
    print(f"  Precision={precision:.2f} Recall={recall:.2f} F1={f1:.2f}")
else:
    print("No test data")

In [ ]:
# Cell 15: Summary
print("=" * 60)
print("E2B EDGE MODEL — BINARY CLASSIFICATION")
print("=" * 60)
print(f"Model:          google/gemma-4-E2B-it (2.3B active, phone-ready)")
print(f"Method:         Unsloth QLoRA, r=8, alpha=8, vision+language")
print(f"Task:           Binary (normal vs abnormal)")
print(f"Dataset:        ICBHI 2017 ({len(train_pairs)} train, balanced)")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"Training time:  {train_time/60:.1f} min")
print(f"Final loss:     {result.training_loss:.4f}")
print(f"Adapter:        {ADAPTER_NAME}")
print()
print("Deploy: Export via LiteRT-LM for phone, 50+ tok/s, 2.58GB disk")
print("=" * 60)

In [ ]:
# Cell 16: DEMO — pick best samples and run end-to-end
import shutil

DEMO_DIR = Path("/kaggle/working/demo_samples")
DEMO_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("DEMO: End-to-end breath sound analysis")
print("=" * 60)

# Pick one abnormal case that PASSed (wheeze or crackle)
demo_cases = []
for pair in test_pairs:
    spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": pair["instruction"]}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=input_text, images=[spec_img], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = extract_json(pred_text)
    exp = json.loads(pair["response"])
    if pred and pred.get("abnormal") == exp.get("abnormal"):
        demo_cases.append({
            "label": pair["label"],
            "original": pair.get("original_label", "?"),
            "spectrogram": pair["spectrogram_path"],
            "prediction": pred,
            "raw_output": pred_text,
        })
    # Get 1 abnormal + 1 normal
    labels_found = set(d["label"] for d in demo_cases)
    if "abnormal" in labels_found and "normal" in labels_found:
        break

print("\nDemo samples (model got these RIGHT):\n")
for i, d in enumerate(demo_cases[-2:]):
    print(f"--- Sample {i+1}: {d['label'].upper()} (original: {d['original']}) ---")
    print(f"Spectrogram: {d['spectrogram']}")
    print(f"Model output: {json.dumps(d['prediction'], indent=2)}")
    # Copy spectrogram to demo dir
    dest = DEMO_DIR / f"demo_{d['label']}_{d['original']}.png"
    shutil.copy2(d["spectrogram"], dest)
    print(f"Saved to: {dest}")
    print()

print("\nDEMO FLOW:")
print("1. Play breathing audio clip to audience")
print("2. Show spectrogram image on screen")
print("3. Feed to Malaika (Gemma 4 + LoRA adapter)")
print("4. Model returns: abnormal=true → triggers IMCI assessment")
print("5. IMCI engine: abnormal breath sounds → classify breathing → recommend action")
print()
print(f"Demo files saved to: {DEMO_DIR}")